# 04 - Build an RDF Graph for a Phenotypic Annotation

## Learning objectives
- Load `hp.owl` and check that `HP:0410050` exists in the ontology.
- Create a custom RDF graph with classes and individuals.
- Model a phenotypic annotation using a blank node and data properties.
- Serialize the resulting graph in Turtle format.

In [2]:
from pathlib import Path
from rdflib import Graph, Namespace, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, XSD

In [3]:
# Resolve project root so the notebook works from project root and from notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "hp.owl").exists():
    project_root = cwd
elif (cwd.parent / "data" / "hp.owl").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find data/hp.owl from current working directory.")

hp_owl_path = project_root / "data" / "hp.owl"

# Load the ontology to verify the HPO class URI we want to use.
hp_graph = Graph()
hp_graph.parse(hp_owl_path)
print(f"Loaded hp.owl triples: {len(hp_graph):,}")

Loaded hp.owl triples: 905,482


In [4]:
# Namespaces used in this notebook.
EX = Namespace("http://example.org/kb/")
OBO = Namespace("http://purl.obolibrary.org/obo/")

# HPO class used as phenotypic feature in the requested graph.
hp_0410050 = URIRef(OBO["HP_0410050"])

# Optional sanity check: ensure HP:0410050 is present in hp.owl and has a label.
feature_label = hp_graph.value(hp_0410050, RDFS.label)
if feature_label is None:
    raise ValueError("HP:0410050 not found in hp.owl.")

print(f"HP:0410050 label: {feature_label}")

HP:0410050 label: Decreased level of 1,5 anhydroglucitol in serum


In [5]:
# Build the custom RDF graph requested in the assignment.
g = Graph()
g.bind("ex", EX)
g.bind("obo", OBO)

# Define classes in our local namespace.
Disease = EX.Disease
PHENOTYPIC_ANNOTATION = EX.PHENOTYPIC_ANNOTATION
Phenotypic_feature = EX.Phenotypic_feature

g.add((Disease, RDF.type, RDFS.Class))
g.add((PHENOTYPIC_ANNOTATION, RDF.type, RDFS.Class))
g.add((Phenotypic_feature, RDF.type, RDFS.Class))

# Define properties used by the annotation node.
has_annotation = EX.hasAnnotation
has_phenotypic_feature = EX.hasPhenotypicFeature
source_prop = EX.source
created_by_prop = EX.createdBy
creation_date_prop = EX.creationDate

# Define OMIM:222100 as an individual of type Disease.
omim_222100 = URIRef("http://omim.org/entry/222100")
g.add((omim_222100, RDF.type, Disease))

# Create the requested newly defined individual X as a blank node.
x = BNode()
g.add((x, RDF.type, PHENOTYPIC_ANNOTATION))

# Connect OMIM:222100 to X.
g.add((omim_222100, has_annotation, x))

# Add data properties to X exactly as requested.
g.add((x, source_prop, Literal("PMIID:9357814")))
g.add((x, created_by_prop, Literal("Nicole Vasilevsky")))
g.add((x, creation_date_prop, Literal("2018-02-23", datatype=XSD.date)))

# Connect X to HP:0410050 and type HP:0410050 as Phenotypic_feature in this graph.
g.add((x, has_phenotypic_feature, hp_0410050))
g.add((hp_0410050, RDF.type, Phenotypic_feature))

print(f"Custom graph triples: {len(g)}")

Custom graph triples: 11


In [6]:
# Serialize and inspect the resulting RDF graph in Turtle.
ttl_output = g.serialize(format="turtle")
print(ttl_output)

@prefix ex: <http://example.org/kb/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:Disease a rdfs:Class .

ex:PHENOTYPIC_ANNOTATION a rdfs:Class .

ex:Phenotypic_feature a rdfs:Class .

<http://omim.org/entry/222100> a ex:Disease ;
    ex:hasAnnotation [ a ex:PHENOTYPIC_ANNOTATION ;
            ex:createdBy "Nicole Vasilevsky" ;
            ex:creationDate "2018-02-23"^^xsd:date ;
            ex:hasPhenotypicFeature obo:HP_0410050 ;
            ex:source "PMIID:9357814" ] .

obo:HP_0410050 a ex:Phenotypic_feature .




## Notes
- `x` is a blank node representing a phenotypic annotation instance.
- `HP:0410050` is a class in HPO; in this local graph we also type it as `ex:Phenotypic_feature` to match the requested structure.